# 🇸🇳 AIMS GAAI Exam — Wolof Assistant Fine-Tuning
**Course:** Applied Generative and Agentic AI — Dr. Papa-Séga WADE  
**Model:** Qwen3-0.6B + LoRA (assistant-only -100 masking)  
**Data:** 3 sources — AYA, Soynade, Synthetic Wolof  


## ✅ Step 0 — Check GPU

In [2]:
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — change runtime!')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB' if torch.cuda.is_available() else '')

GPU available: True
Device: Tesla T4
VRAM: 15.6 GB


## 📦 Step 1 — Install dependencies

In [3]:
%%capture
!pip install transformers>=4.45 accelerate>=0.33 datasets>=2.20 peft>=0.12 trl>=0.10 \
    pandas>=2.0 typer>=0.12 huggingface_hub>=0.24 tensorboard>=2.16 \
    matplotlib>=3.8 PyYAML>=6.0 evaluate sacrebleu rouge_score
print('✅ Installation complete')

## 🔐 Step 2 — Hugging Face Login

In [ ]:
# Paste your HF Write token here
HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXX"
HF_USERNAME = "niangmariame513"
MODEL_NAME = "wolof-assistant-qwen3"
REPO_ID = f"{HF_USERNAME}/{MODEL_NAME}"

from huggingface_hub import login
login(token=HF_TOKEN)
print(f' Logged in as: {HF_USERNAME}')
print(f' Model will be pushed to: {REPO_ID}')

 Logged in as: niangmariame513
 Model will be pushed to: niangmariame513/wolof-assistant-qwen3


## 📁 Step 3 — Clone the exam repository

In [5]:
import os

REPO_URL = "https://github.com/papasega/aims_cases_studies_finetuning_exam"
PROJECT_DIR = "/content/aims_exam"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print('Repo already cloned')

os.chdir(PROJECT_DIR)
!ls

Cloning into '/content/aims_exam'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 34 (delta 4), reused 32 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 102.79 KiB | 862.00 KiB/s, done.
Resolving deltas: 100% (4/4), done.
configs        inference_lora_qwen3_wolof.py  src
data	       push_to_hub.py		      templates
evaluation.py  README.md		      train_lora_assistant_only.py
hf_space       requirements.txt


## 📊 Step 4 — Prepare data (3 sources)

In [6]:
import subprocess, os
os.environ['HF_TOKEN'] = HF_TOKEN

!python src/download_datasets.py all \
    --max-aya-examples 500 \
    --max-soynade-examples 500 \
    --validation-ratio 0.05 \
    --eval-ratio 0.05

print('\n Data sources prepared:')
!ls data/
print('\n Splits:')
!ls data/splits/

Loading CohereLabs/aya_dataset Wolof rows...
README.md: 100% 13.7k/13.7k [00:00<00:00, 8.13MB/s]
Saving aya source rows: 500
Loading soynade-research/Wolof-Non-Standard-Orthography rows...
README.md: 100% 3.81k/3.81k [00:00<00:00, 9.27MB/s]
Saving soynade source rows: 500
Saved synth rows: 5300
Saved chat file for aya: 500 rows -> /content/aims_exam/data/chat_aya.json
Saved aya train: 450 rows -> /content/aims_exam/data/splits/chat_aya_train.json
Saved aya validation: 25 rows -> /content/aims_exam/data/splits/chat_aya_validation.json
Saved aya eval: 25 rows -> /content/aims_exam/data/splits/chat_aya_eval.json
Saved chat file for soynade: 500 rows -> /content/aims_exam/data/chat_soynade.json
Saved soynade train: 450 rows -> /content/aims_exam/data/splits/chat_soynade_train.json
Saved soynade validation: 25 rows -> /content/aims_exam/data/splits/chat_soynade_validation.json
Saved soynade eval: 25 rows -> /content/aims_exam/data/splits/chat_soynade_eval.json
Saved chat file for synth: 530

In [7]:
# Verify all 3 sources are present
import json

sources = {
    'AYA (general)': 'data/splits/chat_aya_train.json',
    'Soynade (domain)': 'data/splits/chat_soynade_train.json',
    'Synthetic (instructions)': 'data/splits/chat_synth_train.json'
}

for name, path in sources.items():
    with open(path) as f:
        data = json.load(f)
    print(f' {name}: {len(data)} training examples')

with open('data/splits/eval_all.jsonl') as f:
    eval_count = sum(1 for _ in f)
print(f'\n Eval set: {eval_count} examples')

 AYA (general): 450 training examples
 Soynade (domain): 450 training examples
 Synthetic (instructions): 4770 training examples

 Eval set: 315 examples


In [9]:
!pip install torchao==0.16.0 -q
print(" torchao fixed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.9 MB/s eta 0:00:00
 torchao fixed


## 🏋️ Step 5 — Train the LoRA adapter

> This uses assistant-only `-100` masking: system/user tokens are ignored, only assistant tokens are learned.

In [10]:

!python train_lora_assistant_only.py \
    --config configs/wolof_training_config.yml \
    --model-choice qwen

Skipping import of cpp extensions due to incompatible torch version 2.11.0+cu128 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
Assistant-only training enabled.
Labels are -100 for system/user/padding tokens; loss is only on assistant outputs.
Training config: /content/aims_exam/configs/wolof_training_config.yml
Model choice: qwen
Base model: Qwen/Qwen3-0.6B
Training data: YAML training_data entries
Output directory: /content/aims_exam/outputs/qwen3_0_6b_wolof_lora
Loaded 450 rows from config source aya: /content/aims_exam/data/splits/chat_aya_train.json
Loaded 450 rows from config source soynade: /content/aims_exam/data/splits/chat_soynade_train.json
Loaded 4770 rows from config source synth: /content/aims_exam/data/splits/chat_synth_train.json
Tokenized examples: kept=5670, skipped=0, avg_total=129.0, avg_supervised=33.9, max_total=1024, max_supervised=1024
Loaded 25 rows from config source aya: /content/aims_exam/data/splits

In [11]:
# Check that adapter was saved
!ls outputs/qwen3_0_6b_wolof_lora/
print('')
!ls outputs/qwen3_0_6b_wolof_lora/best_adapter/ 2>/dev/null || echo 'best_adapter not found, checking latest...'
!ls outputs/qwen3_0_6b_wolof_lora/latest_adapter/ 2>/dev/null

adapter_config.json	   checkpoint-2000		 README.md
adapter_model.safetensors  checkpoint-2100		 tokenizer_config.json
best_adapter		   checkpoint-2127		 tokenizer.json
chat_template.jinja	   checkpoint_eval_results.json  training_curves.json
checkpoint-1800		   latest_adapter		 training_history.json
checkpoint-1900		   plots

adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json
adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json


## 📈 Step 6 — Evaluate (BLEU, ROUGE, F1, Exact Match)

In [12]:
!python evaluation.py \
    --data data/splits/eval_all.jsonl \
    --generate \
    --model-choice qwen \
    --adapter auto

Skipping import of cpp extensions due to incompatible torch version 2.11.0+cu128 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 311/311 [00:00<00:00, 730.18it/s]
Generating prediction 1/315...
Generating prediction 2/315...
Generating prediction 3/315...
Generating prediction 4/315...
Generating prediction 5/315...
Generating prediction 6/315...
Generating prediction 7/315...
Generating prediction 8/315...
Generating prediction 9/315...
Generating prediction 10/315...
Generating prediction 11/315...
Generating prediction 12/315...
Generating prediction 13/315...
Generating prediction 14/315...
Generating prediction 15/315...
Generating prediction 16/315...
Generating prediction 17/315...
Generating prediction 18/315...
Generating prediction 19/315...
Generating prediction 20/315...
Generating prediction 21/315...
Generating prediction 22/315.

## 💬 Step 7 — Test inference locally

In [13]:
!python inference_lora_qwen3_wolof.py \
    --model-choice qwen \
    --adapter auto \
    --category education \
    --instruction "Ana nga?"

Skipping import of cpp extensions due to incompatible torch version 2.11.0+cu128 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
Model choice: qwen
Base model: Qwen/Qwen3-0.6B
Adapter: /content/aims_exam/outputs/qwen3_0_6b_wolof_lora/best_adapter
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 311/311 [00:00<00:00, 690.02it/s]

=== Prompt ===
Category: education
Instruction: Ana nga?

=== Model answer ===
Waw, Ana nga def? Ngii ka yendoo jàmm, mën na gox u die'g.


In [14]:
!python inference_lora_qwen3_wolof.py \
    --model-choice qwen \
    --adapter auto \
    --category education \
    --instruction "Yal na Yàlla baal ma."

Skipping import of cpp extensions due to incompatible torch version 2.11.0+cu128 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
Model choice: qwen
Base model: Qwen/Qwen3-0.6B
Adapter: /content/aims_exam/outputs/qwen3_0_6b_wolof_lora/best_adapter
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 311/311 [00:00<00:00, 640.97it/s]

=== Prompt ===
Category: education
Instruction: Yal na Yàlla baal ma.

=== Model answer ===
Demca ne 'Yàlla' moo bëgg a dem lopitaan.


## 🚀 Step 8 — Push adapter to Hugging Face Hub

In [15]:
!python push_to_hub.py \
    --repo-id {REPO_ID} \
    --model-choice qwen \
    --adapter-dir auto \
    --checkpoint best \
    --public

print(f'\n Model pushed to: https://huggingface.co/{REPO_ID}')

Authenticated Hugging Face user: niangmariame513
Model choice: qwen
Base model: Qwen/Qwen3-0.6B
Selected adapter source (best_adapter): /content/aims_exam/outputs/qwen3_0_6b_wolof_lora/best_adapter
Clean upload folder: /content/aims_exam/outputs/qwen3_0_6b_wolof_lora/hub_upload_adapter
Repository visibility set to public.
Uploading clean adapter files only.
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...ad_adapter/tokenizer.json: 100% 11.4M/11.4M [00:00<?, ?B/s]

Processing Files (1 / 2)      :  22% 11.4M/51.8M [00:00<00:01, 22.0MB/s,   ???B/s  ]


  ...adapter_model.safetensors:   1% 560k/40.4M [00:00<?, ?B/s]

  ...ad_adapter/tokenizer.json: 100% 11.4M/11.4M [00:00<?, ?B/s]


Processing Files (1 / 2)      :  23% 12.0M/51.8M [00:00<00:02, 13.9MB/s, 1.15MB/s  ]
New Data Upload               :   1% 560k/40.4M [00:00<00:54, 727kB/s, 54.7kB/s  ]

  ...ad_adapter/tokenizer.json: 100% 11

## 📋 Step 9 — Save evaluation results for your report

In [16]:
import os, glob

# Show all output files
print('=== Output files ===')
for f in glob.glob('outputs/**/*', recursive=True):
    if os.path.isfile(f) and not f.endswith(('.bin', '.safetensors', '.pt')):
        print(f)

print('\n=== Training curves ===')
for f in glob.glob('outputs/**/*.json', recursive=True):
    print(f)

=== Output files ===
outputs/tensorboard/qwen3_0_6b_wolof_lora/events.out.tfevents.1781193620.c55baa76f188.6790.0
outputs/qwen3_0_6b_wolof_lora/chat_template.jinja
outputs/qwen3_0_6b_wolof_lora/adapter_config.json
outputs/qwen3_0_6b_wolof_lora/training_history.json
outputs/qwen3_0_6b_wolof_lora/tokenizer.json
outputs/qwen3_0_6b_wolof_lora/checkpoint_eval_results.json
outputs/qwen3_0_6b_wolof_lora/training_curves.json
outputs/qwen3_0_6b_wolof_lora/tokenizer_config.json
outputs/qwen3_0_6b_wolof_lora/README.md
outputs/qwen3_0_6b_wolof_lora/checkpoint-2100/chat_template.jinja
outputs/qwen3_0_6b_wolof_lora/checkpoint-2100/adapter_config.json
outputs/qwen3_0_6b_wolof_lora/checkpoint-2100/rng_state.pth
outputs/qwen3_0_6b_wolof_lora/checkpoint-2100/tokenizer.json
outputs/qwen3_0_6b_wolof_lora/checkpoint-2100/trainer_state.json
outputs/qwen3_0_6b_wolof_lora/checkpoint-2100/tokenizer_config.json
outputs/qwen3_0_6b_wolof_lora/checkpoint-2100/README.md
outputs/qwen3_0_6b_wolof_lora/checkpoint-2127

In [17]:
# Plot training loss curve
import json, matplotlib.pyplot as plt, glob

history_files = glob.glob('outputs/**/training_history.json', recursive=True)
if history_files:
    with open(history_files[0]) as f:
        history = json.load(f)

    if 'train_loss' in history:
        plt.figure(figsize=(10,4))
        plt.subplot(1,2,1)
        plt.plot(history['train_loss'], label='Train Loss')
        plt.title('Training Loss')
        plt.xlabel('Step')
        plt.ylabel('Loss')
        plt.legend()

        if 'eval_loss' in history:
            plt.subplot(1,2,2)
            plt.plot(history['eval_loss'], label='Eval Loss', color='orange')
            plt.title('Validation Loss')
            plt.xlabel('Epoch')
            plt.legend()

        plt.tight_layout()
        plt.savefig('outputs/training_curves.png', dpi=150)
        plt.show()
        print(' Curves saved to outputs/training_curves.png')
else:
    print('No training history file found yet')

## 📥 Step 10 — Download adapter for local use (optional)

In [18]:
# Zip the best adapter to download
import shutil

adapter_dir = 'outputs/qwen3_0_6b_wolof_lora/best_adapter'
if not os.path.exists(adapter_dir):
    adapter_dir = 'outputs/qwen3_0_6b_wolof_lora/latest_adapter'

shutil.make_archive('/content/wolof_lora_adapter', 'zip', adapter_dir)
print(' Adapter zipped: /content/wolof_lora_adapter.zip')
print('Download it from the Files panel on the left (folder icon)')

 Adapter zipped: /content/wolof_lora_adapter.zip
Download it from the Files panel on the left (folder icon)
